# End-to-End PyTorch Workflow

Every DL project follows the same six steps. Learning this pattern once means you can attack any new problem without starting from scratch.

```
1. Get and prepare data
2. Build a model
3. Choose loss function and optimizer
4. Train
5. Evaluate and improve
6. Save and reload the model
```

We'll apply all six steps to a single problem: multi-class classification on a 2D toy dataset, keeping the data simple so the focus stays on the workflow pattern.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pathlib

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## Step 1 — Get and Prepare Data

In any real project:
- Download/collect raw data
- Explore it (shape, class balance, missing values)
- Clean and preprocess
- Split into train / val / test
- Wrap in `Dataset` and `DataLoader`

In [ ]:
# 1a. Create data
X_np, y_np = make_blobs(n_samples=1000, centers=4, n_features=2,
                        cluster_std=1.5, random_state=42)

# 1b. Explore
print(f'Samples: {len(X_np)}  Features: {X_np.shape[1]}  Classes: {len(set(y_np))}')
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='tab10', alpha=0.6, s=15)
plt.title('Raw data')
plt.colorbar(label='class')
plt.show()

In [ ]:
# 1c. Preprocess — scale features to zero mean, unit std
scaler = StandardScaler()
X_np   = scaler.fit_transform(X_np)   # fit on ALL data here; in real projects fit on train only

# 1d. Split — train / val / test
X_temp, X_test, y_temp, y_test = train_test_split(X_np, y_np, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15, random_state=42)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

In [ ]:
# 1e. Wrap in DataLoader
def make_loader(X, y, shuffle=False, batch_size=32):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)

## Step 2 — Build a Model

Start simple. Add complexity only if performance demands it.

In [ ]:
class BlobClassifier(nn.Module):
    def __init__(self, in_features, hidden_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )
    def forward(self, x): return self.net(x)


model = BlobClassifier(in_features=2, hidden_size=32, num_classes=4).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 3 — Loss Function and Optimizer

| Task | Loss | Output activation |
|---|---|---|
| Regression | MSELoss | none |
| Binary classification | BCEWithLogitsLoss | none (sigmoid inside loss) |
| Multi-class classification | CrossEntropyLoss | none (softmax inside loss) |

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Step 4 — Train

Track both train and val loss every epoch. Diverging val loss = overfitting.

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += criterion(logits, yb).item()
        correct    += (logits.argmax(1) == yb).sum().item()
        n          += yb.size(0)
    return total_loss / len(loader), correct / n


history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(30):
    tl = train_epoch(model, train_loader, optimizer, criterion)
    vl, va = eval_epoch(model, val_loader, criterion)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(va)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}  train_loss={tl:.4f}  val_loss={vl:.4f}  val_acc={va:.4f}')

In [ ]:
# Plot loss curves — most important diagnostic tool
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set_title('Loss curves')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['val_acc'])
ax2.set_title('Val accuracy')
ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## Step 5 — Evaluate and Improve

**Evaluate** on the held-out test set only once — it's your final honest score.

**Common issues and fixes:**

| Symptom | Cause | Fix |
|---|---|---|
| train_loss high, val_loss high | underfitting | bigger model, more epochs, lower LR |
| train_loss low, val_loss high | overfitting | more data, Dropout, weight decay, early stopping |
| val_loss spiky | LR too high | reduce LR or add scheduler |
| loss NaN | exploding gradients | clip_grad_norm_, lower LR |

In [ ]:
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f'Test loss: {test_loss:.4f}  Test accuracy: {test_acc:.4f}')

In [ ]:
# Visualize decision boundary
import numpy as np

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid   = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    preds = model(grid).argmax(1).cpu().numpy().reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, preds, alpha=0.3, cmap='tab10')
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='tab10', edgecolors='k', s=20)
plt.title('Decision boundary on test set')
plt.show()

## Step 6 — Save and Reload

Always save `state_dict`, not the full model — `state_dict` is just the weights, independent of the class definition.

In [ ]:
# Save
save_dir = pathlib.Path('models')
save_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), save_dir / 'blob_classifier.pth')
print('Saved to models/blob_classifier.pth')

# Reload
loaded = BlobClassifier(in_features=2, hidden_size=32, num_classes=4)
loaded.load_state_dict(torch.load(save_dir / 'blob_classifier.pth', map_location='cpu'))
loaded.eval()

# Verify — same predictions
xb, yb = next(iter(test_loader))
with torch.no_grad():
    orig_pred   = model.cpu()(xb).argmax(1)
    loaded_pred = loaded(xb).argmax(1)
print('Predictions match:', (orig_pred == loaded_pred).all().item())

## Workflow Checklist

Use this for every new DL project:

- [ ] Data split into **train / val / test** (val for tuning, test for final honest score)
- [ ] Preprocessing fitted on **train only**, applied to val/test
- [ ] `model.train()` before training loop, `model.eval()` before evaluation
- [ ] `optimizer.zero_grad()` before `loss.backward()`
- [ ] Loss curves plotted — confirm both curves decrease and don't diverge
- [ ] Final evaluation on test set done **once**
- [ ] Model saved with `state_dict`